# Disaster Tweet Classification: Model 5 — Subword Character n-grams TF-IDF + Balanced Logistic Regression


In [ ]:
import os
import gdown

DATA_DIR = "dataset"
os.makedirs(DATA_DIR, exist_ok=True)

GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"

train_path = os.path.join(DATA_DIR, "train.parquet")
val_path = os.path.join(DATA_DIR, "validation.parquet")
test_path = os.path.join(DATA_DIR, "test.parquet")

# Download from Google Drive if files are not already present
if not (os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path)):
    print("[+] Downloading HumAID dataset from Google Drive...")
    gdown.download_folder(url=GDRIVE_URL, output=DATA_DIR, quiet=False, use_cookies=False)


In [ ]:
import os
import re
import html
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='muted')

# Auto-detect data path (Local workspace vs Kaggle environment)
DATA_DIR = Path("dataset")
if not DATA_DIR.exists():
    kaggle_paths = list(Path("/kaggle/input").glob("**/train.parquet"))
    if kaggle_paths:
        DATA_DIR = kaggle_paths[0].parent
    else:
        DATA_DIR = Path("dataset_csv")

print(f"[+] Active dataset directory: {DATA_DIR}")

if (DATA_DIR / "train.parquet").exists():
    train_df = pd.read_parquet(DATA_DIR / "train.parquet")
    val_df = pd.read_parquet(DATA_DIR / "validation.parquet")
    test_df = pd.read_parquet(DATA_DIR / "test.parquet")
else:
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "validation.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train Split : {train_df.shape} ({len(train_df):,} samples)")
print(f"Val Split   : {val_df.shape} ({len(val_df):,} samples)")
print(f"Test Split  : {test_df.shape} ({len(test_df):,} samples)")


In [ ]:
# --------------------------------------------------------------------------
# 📊 Dataset Insights & EDA Visualizations
# --------------------------------------------------------------------------
def plot_dataset_insights(train_df, val_df, test_df):
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # 1. Class Distribution Across Splits
    classes = sorted(train_df['class_label'].unique())
    dist_df = pd.DataFrame(index=classes)
    dist_df['Train'] = train_df['class_label'].value_counts()
    dist_df['Val'] = val_df['class_label'].value_counts()
    dist_df['Test'] = test_df['class_label'].value_counts()
    dist_df = dist_df.sort_values(by='Train', ascending=True)
    
    clean_labels = [c.replace('_', ' ').title() for c in dist_df.index]
    dist_df.index = clean_labels
    dist_df.plot(kind='barh', stacked=True, ax=axes[0], color=['#2b5c8f', '#e67e22', '#27ae60'], edgecolor='black', alpha=0.85)
    axes[0].set_title('Class Distribution Across Splits (HumAID 10 Classes)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Number of Tweets')
    axes[0].legend(title='Split', frameon=True)
    
    # 2. Tweet Length Distribution
    for name, df, col in [('Train', train_df, '#2b5c8f'), ('Val', val_df, '#e67e22'), ('Test', test_df, '#27ae60')]:
        word_lens = df['tweet_text'].apply(lambda x: len(str(x).split()))
        sns.kdeplot(word_lens, ax=axes[1], label=f"{name} (mean={word_lens.mean():.1f} words)", color=col, fill=True, alpha=0.25)
        
    axes[1].set_title('Tweet Word Count Distribution (KDE)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Word Count per Tweet')
    axes[1].set_ylabel('Density')
    axes[1].set_xlim(0, 50)
    axes[1].legend(title='Split', frameon=True)
    
    plt.tight_layout()
    plt.show()

plot_dataset_insights(train_df, val_df, test_df)


In [ ]:
# --------------------------------------------------------------------------
# Preprocessing Cleaners (Standard, Light, Raw, Aggressive)
# --------------------------------------------------------------------------
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not", "'re": " are",
    "'s": " is", "'d": " would", "'ll": " will", "'t": " not", "'ve": " have",
    "'m": " am", "i'm": "i am", "we're": "we are", "they're": "they are",
    "it's": "it is", "there's": "there is", "that's": "that is", "what's": "what is"
}

DISASTER_SLANG = {
    r"\bpls\b": "please", r"\bplz\b": "please", r"\bthx\b": "thanks",
    r"\bu\b": "you", r"\bur\b": "your", r"\br\b": "are",
    r"\bw/\b": "with", r"\bw/o\b": "without", r"\bb4\b": "before",
    r"\bmsg\b": "message", r"\binfo\b": "information", r"\bemerg\b": "emergency",
    r"\bevac\b": "evacuation", r"\bevacs\b": "evacuations", r"\bvicts\b": "victims",
    r"\bgov\b": "government", r"\bdept\b": "department", r"\bvol\b": "volunteer"
}

EMOJI_TRANSLATIONS = {
    "🙏": " prayer support ", "💔": " heartbreak grief ", "❤️": " love sympathy ",
    "🚨": " emergency warning alert ", "⚠️": " danger warning caution ",
    "🔥": " fire wildfire disaster ", "🌊": " flood tsunami water surge ",
    "🌧️": " rain storm hurricane ", "🌪️": " tornado storm ", "⚡": " storm lightning ",
    "😢": " crying sorrow sadness ", "😭": " weeping tragedy ", "🕯️": " mourning memorial ",
    "🆘": " urgent help request emergency ", "🏠": " shelter home house ", "🏥": " hospital medical clinic "
}

def split_camel_case(text: str) -> str:
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

def clean_raw(text: str) -> str:
    return str(text) if text is not None else ""

def clean_light(text: str) -> str:
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_standard(text: str) -> str:
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    for em, rep in EMOJI_TRANSLATIONS.items(): text = text.replace(em, rep)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', lambda m: split_camel_case(m.group(1)), text)
    text = text.lower()
    for c, exp in CONTRACTIONS.items(): text = text.replace(c, exp)
    for p, rep in DISASTER_SLANG.items(): text = re.sub(p, rep, text, flags=re.IGNORECASE)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\brt\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_aggressive(text: str) -> str:
    import string
    text = clean_standard(text)
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    stopwords = {'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'it', 'this', 'that', 'from', 'as', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'but', 'not', 'so', 'we', 'i', 'you', 'they', 'he', 'she', 'my', 'your', 'our', 'their'}
    words = [w for w in text.split() if w not in stopwords and len(w) > 2]
    return ' '.join(words)


In [ ]:
# --------------------------------------------------------------------------
# 📈 Comprehensive Evaluation Suite & Metric Plotting Functions
# --------------------------------------------------------------------------
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def plot_comprehensive_evaluation(y_true, y_pred, class_names, model_title="Model", y_train=None):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    w_p, w_r, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    p_per, r_per, f1_per, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    
    clean_classes = [c.replace('_', ' ').title() for c in class_names]
    
    # Print Terminal Report
    print("=" * 75)
    print(f"               {model_title.upper()} — EVALUATION SUMMARY")
    print("=" * 75)
    print(f"Accuracy          : {acc * 100:.2f}%")
    print(f"Macro F1-Score    : {macro_f1 * 100:.2f}%  <-- [Primary Optimization Metric]")
    print(f"Weighted F1-Score : {w_f1 * 100:.2f}%")
    print(f"Macro Precision   : {macro_p * 100:.2f}%")
    print(f"Macro Recall      : {macro_r * 100:.2f}%")
    print("=" * 75)
    print("\nDetailed Per-Class Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=clean_classes, digits=4, zero_division=0))
    
    # 1. Dual Confusion Matrices (Normalized & Raw Counts)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0], xticklabels=clean_classes, yticklabels=clean_classes)
    axes[0].set_title(f"{model_title} — Normalized Confusion Matrix (Macro-F1: {macro_f1*100:.2f}%)", fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Predicted Class', fontsize=11)
    axes[0].set_ylabel('True Class', fontsize=11)
    axes[0].tick_params(axis='x', rotation=45)
    
    cm_raw = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=clean_classes, yticklabels=clean_classes)
    axes[1].set_title(f"{model_title} — Raw Counts Confusion Matrix", fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Predicted Class', fontsize=11)
    axes[1].set_ylabel('True Class', fontsize=11)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Per-Class Triple Bar Chart (Precision, Recall, F1)
    df_metrics = pd.DataFrame({
        'Class': clean_classes,
        'Precision (%)': p_per * 100,
        'Recall (%)': r_per * 100,
        'F1-Score (%)': f1_per * 100
    }).sort_values(by='F1-Score (%)', ascending=True)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    df_metrics.plot(x='Class', y=['Precision (%)', 'Recall (%)', 'F1-Score (%)'], kind='barh', ax=ax, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.9)
    ax.set_title(f"{model_title} — Per-Class Precision, Recall & F1-Score Breakdown", fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Score (%)', fontsize=12)
    ax.set_xlim(0, 105)
    ax.legend(frameon=True, fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # 3. Class Imbalance vs F1-Score Correlation (if y_train is provided)
    if y_train is not None:
        train_counts = pd.Series(y_train).value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.regplot(x=train_counts.values, y=f1_per * 100, ax=ax, scatter_kws={'s': 100, 'color': '#2c3e50'}, line_kws={'color': '#e74c3c', 'linestyle': '--'})
        ax.set_title('Training Sample Count vs Test Per-Class F1-Score (Imbalance Robustness)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Number of Training Samples (Class Frequency)', fontsize=11)
        ax.set_ylabel('Test F1-Score (%)', fontsize=11)
        for i, txt in enumerate(clean_classes):
            ax.annotate(txt, (train_counts.values[i], f1_per[i] * 100 + 1.0), fontsize=9)
        plt.tight_layout()
        plt.show()


## 1. Character n-gram Extraction & Model Fitting


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df['clean_text'] = train_df['tweet_text'].apply(clean_standard)
val_df['clean_text'] = val_df['tweet_text'].apply(clean_standard)
test_df['clean_text'] = test_df['tweet_text'].apply(clean_standard)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['class_label'])
y_val = label_encoder.transform(val_df['class_label'])
y_test = label_encoder.transform(test_df['class_label'])
classes = list(label_encoder.classes_)

char_tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=30000, sublinear_tf=True)
X_train = char_tfidf.fit_transform(train_df['clean_text'])
X_val = char_tfidf.transform(val_df['clean_text'])
X_test = char_tfidf.transform(test_df['clean_text'])
print("Char TF-IDF matrix shape:", X_train.shape)

clf = LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_val_pred = clf.predict(X_val)
_, _, val_f1, _ = precision_recall_fscore_support(y_val, y_val_pred, average='macro', zero_division=0)
print(f"[+] Validation Macro-F1: {val_f1*100:.2f}%")


## 2. Evaluation Suite & Plots


In [ ]:
y_test_pred = clf.predict(X_test)
plot_comprehensive_evaluation(y_test, y_test_pred, classes, model_title="Model 5 (Char TF-IDF + LR)", y_train=y_train)
